# DEE Test T³/S³ — Autovectores y 3-esfera

Este notebook cierra el paquete numérico con dos tests complementarios:

**Parte A — Test S³:** verificamos que en la 3-esfera el Laplaciano DEE produce tripleto (en lugar de sexteto) como predice la curvatura positiva, y que el óptimo de jitter se preserva. Si S³ se comporta distinto de T³ en la forma esperada, el test es discriminativo.

**Parte B — Autovectores en T³:** no basta con que los autovalores sean los correctos; los autovectores deben ser las ondas planas e^(i·2π·n·x) del toro. Verificamos que los modos del grafo tienen soporte Fourier concentrado en |n|²=1 para los primeros seis.

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
import time

PLOT_DIR = 'plots_T3_S3_final'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name, fig=None, dpi=120):
    if fig is None:
        fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → {path}')

print(f'Plots se guardarán en: {PLOT_DIR}/')

## 1. Funciones base (T³ y S³)

In [ ]:
# ---------- T³ ----------
def periodic_distance_matrix(points, L=1.0):
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def grid_perturbed_T3(N_target, jitter_fraction, seed=42):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

# ---------- S³ ----------
def sample_S3_quasi_regular(N_target, jitter_fraction=0.1, seed=42):
    """
    Muestra cuasi-regular de S³ ⊂ R⁴.
    
    Estrategia: generamos una distribución aproximadamente uniforme en S³
    usando spiral/Fibonacci extendido, y le agregamos jitter tangente.
    """
    rng = np.random.default_rng(seed)
    
    # Muestra uniforme usando normalización de gaussianas (método estándar)
    x = rng.normal(size=(N_target, 4))
    x = x / np.linalg.norm(x, axis=1, keepdims=True)
    
    # Para tener "cuasi-regular", repelemos puntos cercanos (Lloyd simplificado)
    # Hacemos unas pocas iteraciones de relajación para reducir clumping
    for _ in range(3):
        # Encontrar los 10 vecinos más cercanos y repeler un poco
        D = np.arccos(np.clip(x @ x.T, -1, 1))
        np.fill_diagonal(D, np.inf)
        # Vecinos más cercanos
        nearest = np.argpartition(D, 10, axis=1)[:, :10]
        
        displacements = np.zeros_like(x)
        for i in range(N_target):
            for j in nearest[i]:
                # Dirección de repulsión: de j hacia i, proyectada sobre tangente de S³
                direction = x[i] - x[j]
                # Proyectar a tangente en x[i]: quitar componente radial
                direction -= np.dot(direction, x[i]) * x[i]
                d = D[i, j]
                if d > 0:
                    displacements[i] += direction / (d**2 * 100)  # factor pequeño
        
        x = x + displacements
        x = x / np.linalg.norm(x, axis=1, keepdims=True)
    
    # Ahora agregamos jitter tangente controlable
    if jitter_fraction > 0:
        # Típico spacing: ~(Area/N)^(1/3) = (2π²/N)^(1/3) pero como estamos normalizados
        # en términos de distancia geodésica, usamos ~π·N^(-1/3) como escala
        typical_spacing = np.pi * N_target**(-1/3)
        jitter_amount = jitter_fraction * typical_spacing
        
        # Jitter: empujamos cada punto en dirección aleatoria tangente
        tangent_dirs = rng.normal(size=(N_target, 4))
        # Proyectar a tangente (quitar componente radial)
        tangent_dirs -= np.sum(tangent_dirs * x, axis=1, keepdims=True) * x
        tangent_dirs = tangent_dirs / np.linalg.norm(tangent_dirs, axis=1, keepdims=True)
        
        x = x + jitter_amount * tangent_dirs
        x = x / np.linalg.norm(x, axis=1, keepdims=True)
    
    return x

def geodesic_distance_S3(points):
    dots = points @ points.T
    dots = np.clip(dots, -1.0, 1.0)
    return np.arccos(dots)

# ---------- Laplaciano (común) ----------
def build_laplacian_from_distances(D_mat, k_neighbors=30, alpha=2.0):
    N = len(D_mat)
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    neighbor_idx = np.argpartition(D_search, k_neighbors, axis=1)[:, :k_neighbors]
    
    rows, cols, vals = [], [], []
    for i in range(N):
        for j in neighbor_idx[i]:
            d = D_mat[i, j]
            if d > 0:
                w = 1.0 / d**alpha
                rows.append(i); cols.append(j); vals.append(w)
    
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    degrees = np.array(W.sum(axis=1)).flatten()
    return diags(degrees) - W

def get_eigen(L_sparse, n_eig=20):
    try:
        eigs, vecs = eigsh(L_sparse, k=n_eig, which='SM', tol=1e-8)
    except Exception:
        # fallback denso
        L_dense = L_sparse.toarray()
        eigs_all, vecs_all = np.linalg.eigh(L_dense)
        eigs = eigs_all[:n_eig]
        vecs = vecs_all[:, :n_eig]
    # Ordenar
    idx = np.argsort(eigs)
    return eigs[idx], vecs[:, idx]

def dispersion(eigs, i_start, i_end):
    """Dispersión relativa para un grupo de autovalores contiguos."""
    group = eigs[i_start:i_end]
    return (group.max() - group.min()) / group.max()

print('Funciones listas.')

## 2. Parte A — Test S³: tripleto vs sexteto

En S³ (3-esfera) los armónicos esféricos se organizan en multipletes de tamaño (n+1)² para n=0,1,2,...:
- n=0: singlete (modo constante)
- n=1: **cuadruplete** (4 modos con mismo autovalor)
- n=2: 9-plete

Para S³ la degeneración del primer nivel excitado es 4 (no 3 como dije antes, me corrijo). Los autovalores del Laplaciano en S³ de radio 1 son n(n+2) para n=0,1,2,...:
- λ₀ = 0
- λ₁ = 3 (multiplicidad 4)
- λ₂ = 8 (multiplicidad 9)

Entonces en el plot esperamos:
- 4 autovalores iguales (el "cuadruplete" que es la firma de S³)
- Luego 9 autovalores iguales
- Ratio λ_5 / λ_1 = 8/3 ≈ 2.67

In [ ]:
# Barrido de jitter en S³
jitter_values_S3 = [0.0, 0.1, 0.25, 0.5, 1.0]
seeds = [42, 123, 2024]
N_target_S3 = 1200
k_test = 30

results_S3 = {}

print(f'Test S³: barrido de jitter (N≈{N_target_S3}, k={k_test}, 3 semillas)')
print(f"{'jitter':>8} {'disp cuadrup.':>14} {'disp "sexteto"':>16} {'λ₅/λ₁':>10}")
print('-'*60)

t0 = time.time()
for j in jitter_values_S3:
    disps_quad = []
    disps_sext = []
    ratios = []
    for seed in seeds:
        points = sample_S3_quasi_regular(N_target_S3, jitter_fraction=j, seed=seed)
        D_geo = geodesic_distance_S3(points)
        L_S3 = build_laplacian_from_distances(D_geo, k_neighbors=k_test, alpha=2.0)
        eigs, _ = get_eigen(L_S3, n_eig=15)
        
        # Dispersión del cuadruplete (λ₁ a λ₄)
        disps_quad.append(dispersion(eigs, 1, 5))
        # Dispersión si uno tratara los primeros 6 como si fuera "sexteto"
        disps_sext.append(dispersion(eigs, 1, 7))
        ratios.append(eigs[5]/eigs[1] if eigs[1] > 0 else 0)
    
    results_S3[j] = {
        'disp_quad': np.mean(disps_quad),
        'disp_quad_std': np.std(disps_quad),
        'disp_sext': np.mean(disps_sext),
        'disp_sext_std': np.std(disps_sext),
        'ratio': np.mean(ratios)
    }
    r = results_S3[j]
    print(f'{j:>8.2f} {r["disp_quad"]*100:>10.2f}% ±{r["disp_quad_std"]*100:.1f}  '
          f'{r["disp_sext"]*100:>10.2f}% ±{r["disp_sext_std"]*100:.1f}  '
          f'{r["ratio"]:>10.3f}')

print(f'\nRatio teórico S³: λ₅/λ₁ = 8/3 ≈ 2.667')
print(f'Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# Plot comparativo: cuadruplete vs "sexteto" en S³
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

js = list(results_S3.keys())
quad_means = [results_S3[j]['disp_quad']*100 for j in js]
quad_stds = [results_S3[j]['disp_quad_std']*100 for j in js]
sext_means = [results_S3[j]['disp_sext']*100 for j in js]
sext_stds = [results_S3[j]['disp_sext_std']*100 for j in js]

ax1.errorbar(js, quad_means, yerr=quad_stds, fmt='o-', markersize=11,
             linewidth=2, color='darkblue', capsize=5,
             label='Cuadruplete λ₁...λ₄ (firma S³)')
ax1.errorbar(js, sext_means, yerr=sext_stds, fmt='s--', markersize=11,
             linewidth=2, color='darkred', capsize=5, alpha=0.7,
             label='"Sexteto" λ₁...λ₆ (no debería degenerar)')
ax1.axhline(10, linestyle=':', color='green', alpha=0.5, label='Umbral 10%')
ax1.set_xlabel('Jitter', fontsize=12)
ax1.set_ylabel('Dispersión relativa [%]', fontsize=12)
ax1.set_title('S³: cuadruplete limpio, sexteto roto → firma de curvatura positiva', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Plot de ratios
ratios_S3 = [results_S3[j]['ratio'] for j in js]
ax2.plot(js, ratios_S3, 'o-', markersize=11, linewidth=2, color='darkgreen')
ax2.axhline(8/3, linestyle='--', color='red', alpha=0.6,
            label=f'Teórico S³: λ₅/λ₁ = 8/3 ≈ {8/3:.3f}')
ax2.set_xlabel('Jitter', fontsize=12)
ax2.set_ylabel('λ₅ / λ₁', fontsize=12)
ax2.set_title('Gap espectral entre cuadruplete y 9-plete', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
save_plot('09_S3_cuadruplete_vs_sexteto')
plt.show()

## 3. Parte A — Plot comparativo T³ vs S³ lado a lado

In [ ]:
# Comparación directa T³ vs S³ con jitter óptimo (0.1 o 0.25)
j_opt = 0.1

# T³
points_T3, N_T3 = grid_perturbed_T3(1331, j_opt, seed=42)
D_T3 = periodic_distance_matrix(points_T3, L=1.0)
L_T3 = build_laplacian_from_distances(D_T3, k_neighbors=30, alpha=2.0)
eigs_T3, vecs_T3 = get_eigen(L_T3, n_eig=20)

# S³
points_S3 = sample_S3_quasi_regular(1200, jitter_fraction=j_opt, seed=42)
D_S3 = geodesic_distance_S3(points_S3)
L_S3 = build_laplacian_from_distances(D_S3, k_neighbors=30, alpha=2.0)
eigs_S3, _ = get_eigen(L_S3, n_eig=20)

# Normalizar al primer autovalor no-nulo
eigs_T3_norm = eigs_T3 / eigs_T3[1]
eigs_S3_norm = eigs_S3 / eigs_S3[1]

# Valores teóricos
T3_theoretical = np.sort([nx**2+ny**2+nz**2 for nx in range(-3,4) for ny in range(-3,4) for nz in range(-3,4)])[:20]
# S³: λ_n = n(n+2), multiplicidades (n+1)²
S3_theoretical = [0]  # n=0
for n_quant in range(1, 6):
    mult = (n_quant+1)**2
    lam = n_quant*(n_quant+2)
    S3_theoretical.extend([lam]*mult)
S3_theoretical = np.array(S3_theoretical[:20]) / 3.0  # normalizar al primero

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# T³
ax = axes[0]
ax.plot(np.arange(20), eigs_T3_norm[:20], 'o', color='steelblue',
        markersize=12, label=f'DEE en T³ (jitter={j_opt})')
ax.plot(np.arange(20), T3_theoretical, 'x', color='red', markersize=15,
        markeredgewidth=2.5, label='T³ analítico |n|²')
ax.axvspan(0.5, 6.5, alpha=0.15, color='green', label='Sexteto |n|²=1')
ax.axvspan(6.5, 18.5, alpha=0.08, color='orange', label='12-plete |n|²=2')
ax.set_xlabel('Índice', fontsize=12)
ax.set_ylabel('λ / λ₁', fontsize=12)
ax.set_title('T³ plano — sexteto + 12-plete', fontsize=13)
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)
ax.set_ylim(-0.3, 4.5)

# S³
ax = axes[1]
ax.plot(np.arange(20), eigs_S3_norm[:20], 's', color='crimson',
        markersize=12, label=f'DEE en S³ (jitter={j_opt})')
ax.plot(np.arange(20), S3_theoretical, 'x', color='red', markersize=15,
        markeredgewidth=2.5, label='S³ analítico n(n+2)/3')
ax.axvspan(0.5, 4.5, alpha=0.15, color='blue', label='Cuadruplete n=1')
ax.axvspan(4.5, 13.5, alpha=0.08, color='purple', label='9-plete n=2')
ax.set_xlabel('Índice', fontsize=12)
ax.set_ylabel('λ / λ₁', fontsize=12)
ax.set_title('S³ curvado — cuadruplete + 9-plete', fontsize=13)
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)
ax.set_ylim(-0.3, 4.5)

plt.tight_layout()
save_plot('10_T3_vs_S3_lado_a_lado')
plt.show()

print(f'\nT³ dispersión sexteto (jitter={j_opt}): {dispersion(eigs_T3, 1, 7)*100:.2f}%')
print(f'S³ dispersión cuadruplete (jitter={j_opt}): {dispersion(eigs_S3, 1, 5)*100:.2f}%')
print('\nSi el test funciona, ambos valores deberían ser pequeños (<10%) y las')
print('firmas espectrales (sexteto vs cuadruplete) claramente distintas.')

## 4. Parte B — Análisis de autovectores en T³

Los autovalores solos no son suficientes. Hay que verificar que los **autovectores** del Laplaciano de grafo son las ondas planas del toro:

$$\psi_{\vec{n}}(x) = e^{i \, 2\pi \, \vec{n} \cdot \vec{x}}$$

Para el sexteto |n|²=1, esperamos ondas con vectores ±x̂, ±ŷ, ±ẑ.

**Test:** proyectamos cada autovector del grafo sobre la base de Fourier y verificamos que el peso está concentrado en |n|²=1.

**Definición:** para un autovector v del grafo con soporte en los puntos {x_i}, su proyección Fourier es:

$$\tilde{v}(\vec{n}) = \frac{1}{N} \sum_i v_i \, e^{-i \, 2\pi \, \vec{n} \cdot \vec{x}_i}$$

Si v corresponde al modo n*, entonces |ṽ(n*)|² debería ser cercano a 1 y todos los otros cercanos a 0.

In [ ]:
def fourier_projection(v, points, n_max=3):
    """
    Proyección de Fourier de un vector v sobre ondas planas de T³.
    Retorna un dict {(nx,ny,nz): |peso|²}.
    """
    weights = {}
    N = len(v)
    for nx in range(-n_max, n_max+1):
        for ny in range(-n_max, n_max+1):
            for nz in range(-n_max, n_max+1):
                n_vec = np.array([nx, ny, nz])
                phase = 2 * np.pi * (points @ n_vec)
                basis = np.exp(-1j * phase) / np.sqrt(N)
                projection = np.sum(v * basis) / np.sqrt(N)
                weights[(nx, ny, nz)] = abs(projection)**2
    return weights

# Usar el mismo grafo T³ con jitter=0.1
print('Analizando los primeros 7 autovectores del Laplaciano DEE en T³ (jitter=0.1)...\n')

# vecs_T3 y eigs_T3 ya están calculados
# Para cada uno de los autovectores 0-6, calculamos su proyección Fourier

fourier_analysis = {}
for mode_idx in range(7):
    v = vecs_T3[:, mode_idx]
    # Normalizar (debería ya estar normalizado, pero por si acaso)
    v = v / np.linalg.norm(v)
    
    weights = fourier_projection(v, points_T3, n_max=2)
    fourier_analysis[mode_idx] = {
        'eigenvalue': eigs_T3[mode_idx],
        'weights': weights
    }
    
    # Encontrar los top-3 modos Fourier por peso
    sorted_weights = sorted(weights.items(), key=lambda x: -x[1])
    top3 = sorted_weights[:3]
    
    # Calcular peso total en |n|²=1 (las 6 ondas planas del sexteto)
    weight_sexteto = sum(w for n, w in weights.items()
                         if sum(x**2 for x in n) == 1)
    weight_total = sum(weights.values())
    
    print(f'Modo {mode_idx}: λ = {eigs_T3[mode_idx]:.3f}')
    print(f'   Top-3 ondas planas: {[(n, f"{w:.3f}") for n, w in top3]}')
    print(f'   Peso total en |n|²=1: {weight_sexteto:.3f}  (total norm: {weight_total:.3f})')
    print()

In [ ]:
# Plot: matriz de pesos Fourier para los primeros 7 modos
fig, axes = plt.subplots(1, 7, figsize=(20, 3.5), sharey=True)

# Construir la base ordenada: (0,0,0), luego |n|²=1 (6 modos), luego |n|²=2 (12 modos), ...
n_basis = []
for n2_max in range(5):
    for nx in range(-2, 3):
        for ny in range(-2, 3):
            for nz in range(-2, 3):
                if nx**2+ny**2+nz**2 == n2_max:
                    n_basis.append((nx, ny, nz))
n_basis = n_basis[:25]  # primeros 25

# Labels y colores por |n|²
n2_per_mode = [n[0]**2+n[1]**2+n[2]**2 for n in n_basis]
colors_per_n2 = {0: 'gray', 1: 'green', 2: 'orange', 3: 'purple', 4: 'blue'}
bar_colors = [colors_per_n2.get(n2, 'black') for n2 in n2_per_mode]

for mode_idx in range(7):
    ax = axes[mode_idx]
    weights = fourier_analysis[mode_idx]['weights']
    heights = [weights.get(n, 0) for n in n_basis]
    
    ax.bar(np.arange(len(n_basis)), heights, color=bar_colors)
    ax.set_xticks([])
    ax.set_title(f'Modo {mode_idx}\nλ={eigs_T3[mode_idx]:.2f}', fontsize=10)
    if mode_idx == 0:
        ax.set_ylabel('|ṽ(n)|²', fontsize=10)

# Leyenda
from matplotlib.patches import Patch
legend_items = [Patch(color='gray', label='|n|²=0'),
                Patch(color='green', label='|n|²=1 (sexteto)'),
                Patch(color='orange', label='|n|²=2'),
                Patch(color='purple', label='|n|²=3'),
                Patch(color='blue', label='|n|²=4')]
fig.legend(handles=legend_items, loc='upper center', bbox_to_anchor=(0.5, 0.04), ncol=5)

plt.suptitle('Descomposición Fourier de autovectores DEE en T³ (jitter=0.1)', fontsize=13, y=1.02)
plt.tight_layout()
save_plot('11_autovectores_fourier')
plt.show()

In [ ]:
# Test cuantitativo: ¿qué fracción de la norma de cada autovector vive en el sexteto?
print('Test cuantitativo de localización en Fourier:')
print(f"{'Modo':>5} {'λ':>8} {'peso |n|²=0':>14} {'peso |n|²=1':>14} {'peso |n|²=2':>14} {'peso |n|²>2':>14}")
print('-' * 80)

pesos_sexteto = []
for mode_idx in range(7):
    weights = fourier_analysis[mode_idx]['weights']
    total = sum(weights.values())
    
    w_0 = sum(w for n,w in weights.items() if sum(x**2 for x in n)==0) / total
    w_1 = sum(w for n,w in weights.items() if sum(x**2 for x in n)==1) / total
    w_2 = sum(w for n,w in weights.items() if sum(x**2 for x in n)==2) / total
    w_hi = sum(w for n,w in weights.items() if sum(x**2 for x in n)>2) / total
    
    pesos_sexteto.append(w_1)
    marker = ' ←' if (mode_idx >= 1 and w_1 > 0.80) else ''
    
    print(f'{mode_idx:>5} {eigs_T3[mode_idx]:>8.3f} {w_0:>14.3f} {w_1:>14.3f} '
          f'{w_2:>14.3f} {w_hi:>14.3f}{marker}')

print()
print(f'Modos 1-6 con peso >80% en |n|²=1: {sum(1 for w in pesos_sexteto[1:7] if w > 0.80)}/6')
print(f'Peso medio del sexteto en |n|²=1: {np.mean(pesos_sexteto[1:7]):.3f}')
print()
print('Interpretación:')
print('Si los modos 1-6 tienen peso >80% en |n|²=1, los autovectores SÍ son')
print('las ondas planas de T³. La geometría emerge completa, no solo los autovalores.')

## 5. Síntesis final y empaquetado

In [ ]:
print('='*70)
print('SÍNTESIS — Test T³ + S³ final')
print('='*70)
print()
print('PARTE A — S³ (firma de curvatura positiva):')
for j in sorted(results_S3.keys()):
    r = results_S3[j]
    contrast = r['disp_sext']/r['disp_quad'] if r['disp_quad'] > 0 else float('inf')
    print(f'  jitter={j:.2f}: cuadruplete {r["disp_quad"]*100:.1f}%  '
          f'vs sexteto-bad {r["disp_sext"]*100:.1f}%  (ratio {contrast:.1f}x)')
print()

print('PARTE B — Autovectores T³ (jitter=0.1, N=1331):')
for mode_idx in range(7):
    weights = fourier_analysis[mode_idx]['weights']
    total = sum(weights.values())
    w_1 = sum(w for n,w in weights.items() if sum(x**2 for x in n)==1) / total
    print(f'  Modo {mode_idx}: peso en |n|²=1 = {w_1:.3f}')
print()

# Conclusiones automáticas
S3_ok = results_S3[0.1]['disp_quad'] * 100 < 10
S3_contrast = results_S3[0.1]['disp_sext'] > 1.5 * results_S3[0.1]['disp_quad']
sexteto_fourier_ok = sum(1 for w in pesos_sexteto[1:7] if w > 0.7) >= 5

print('='*70)
print('RESULTADOS')
print('='*70)
print()
print(f'[A1] S³ produce cuadruplete limpio con jitter óptimo:      {"SÍ" if S3_ok else "NO"}')
print(f'[A2] S³ es distinguible de T³ (sexteto vs cuadruplete):    {"SÍ" if S3_contrast else "NO"}')
print(f'[B1] Autovectores T³ son ondas planas (peso >70% en |n|²=1): {"SÍ" if sexteto_fourier_ok else "PARCIAL"}')
print()

if S3_ok and S3_contrast and sexteto_fourier_ok:
    print('→ Paquete numérico CERRADO: DEE produce geometría T³ y S³ correctas,')
    print('  con firmas espectrales distinguibles y autovectores físicamente')
    print('  correctos. Material suficiente para paper corto.')
else:
    print('→ Paquete numérico parcial. Ver qué tests fallaron e iterar.')

In [ ]:
import shutil
shutil.make_archive('plots_T3_S3_final', 'zip', PLOT_DIR)
print(f'\nZip creado: plots_T3_S3_final.zip')

try:
    from google.colab import files
    files.download('plots_T3_S3_final.zip')
    print('→ Descarga iniciada')
except ImportError:
    pass